# Seismic LSTM Autoencoder — Training & Evaluation

Trains the anomaly-detection autoencoder on USGS earthquake catalog for the Indian subcontinent.

**Output:** `models/seismic_ae.pt` — loaded by `backend/app/ml/seismic_autoencoder.py`

## Data
Download 5-year USGS catalog for India bounding box:
```
https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv
  &starttime=2019-01-01&endtime=2024-12-31
  &minlatitude=6&maxlatitude=37
  &minlongitude=68&maxlongitude=98
  &minmagnitude=2.0
```
Save as `data/usgs_india_5yr.csv`.

In [ ]:
import sys
from pathlib import Path

# Add backend to path so we can import app.ml
sys.path.insert(0, str(Path.cwd().parents[1] / 'backend'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from app.ml.seismic_model import SeismicAutoencoder

CATALOG_PATH = Path('../data/usgs_india_5yr.csv')
OUTPUT_PATH  = Path('../models/seismic_ae.pt')
SEQ_LEN = 30
FEATURE_COLS = ['latitude', 'longitude', 'depth', 'magnitude', 'rms', 'gap', 'horizontalError', 'count_30d']

In [ ]:
# ── Load catalog ──────────────────────────────────────────────
df = pd.read_csv(CATALOG_PATH, parse_dates=['time'])
df = df.sort_values('time').reset_index(drop=True)
df['count_30d'] = df.set_index('time')['magnitude'].rolling('30D').count().values
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0.0)
print(f'Loaded {len(df):,} events  ({df.time.min().date()} → {df.time.max().date()})')
df[FEATURE_COLS].describe()

In [ ]:
# ── Build sequences + normalise ───────────────────────────────
from sklearn.preprocessing import StandardScaler

data = df[FEATURE_COLS].values.astype(np.float32)
scaler = StandardScaler().fit(data)
data_scaled = scaler.transform(data)

seqs = np.array([data_scaled[i:i+SEQ_LEN] for i in range(len(data_scaled)-SEQ_LEN)], dtype=np.float32)
print(f'Sequences: {seqs.shape}')  # (N, 30, 8)

In [ ]:
# ── Train ─────────────────────────────────────────────────────
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 50
BATCH  = 64
LR     = 1e-3

model = SeismicAutoencoder(input_size=len(FEATURE_COLS), seq_len=SEQ_LEN)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

tensor  = torch.tensor(seqs)
loader  = DataLoader(TensorDataset(tensor), batch_size=BATCH, shuffle=True)
history = []

for epoch in range(1, EPOCHS+1):
    total = 0.0
    for (batch,) in loader:
        opt.zero_grad()
        loss = loss_fn(model(batch), batch)
        loss.backward()
        opt.step()
        total += loss.item() * len(batch)
    avg = total / len(tensor)
    history.append(avg)
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={avg:.6f}')

plt.plot(history)
plt.xlabel('Epoch'); plt.ylabel('MSE loss'); plt.title('Training loss')
plt.tight_layout(); plt.show()

In [ ]:
# ── Evaluate: reconstruction error distribution ───────────────
model.eval()
with torch.no_grad():
    recon = model(tensor)
    errors = torch.mean((recon - tensor)**2, dim=(1,2)).numpy()

threshold_95 = np.percentile(errors, 95)
print(f'95th pct reconstruction error (anomaly threshold): {threshold_95:.6f}')

plt.hist(errors, bins=100, log=True)
plt.axvline(threshold_95, color='red', label='95th pct')
plt.xlabel('Reconstruction MSE'); plt.ylabel('Count'); plt.legend()
plt.title('Anomaly score distribution'); plt.tight_layout(); plt.show()

In [ ]:
# ── Save weights ──────────────────────────────────────────────
import pickle

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), OUTPUT_PATH)

with open(OUTPUT_PATH.with_suffix('.scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

print(f'Saved model → {OUTPUT_PATH}')
print(f'Copy to /tmp/seismic_ae.pt for inference: cp {OUTPUT_PATH} /tmp/seismic_ae.pt')